# Data Vortex - Round 1: Initial Dataset Inspection

## 1. Overview and Objective
This notebook performs an initial inspection and audit of the raw dataset recovered for the **Data Vortex Round 1** competition:
`data/raw/Social_Engine_Users.csv`.

### Rules & Constraints
- Strictly read-only audit: no modifications, deletions, imputation, or data corrections are made here.
- All observations are documented as **Confirmed Observations**, **Potentially Suspicious Values**, or **Issues Requiring Further Investigation**.

In [ ]:
import os
import csv
import pandas as pd
import numpy as np

# File path specification
DATA_PATH = os.path.join("..", "data", "raw", "Social_Engine_Users.csv")
print(f"Target dataset path: {DATA_PATH}")
print(f"File exists: {os.path.exists(DATA_PATH)}")
print(f"File size: {os.path.getsize(DATA_PATH):,} bytes")

## 2. File Properties, Encoding & Delimiter Sniffing
Inspect the raw binary header to detect character encoding, byte-order marks (BOM), and the CSV dialect delimiter.

In [ ]:
# Read raw sample bytes
with open(DATA_PATH, "rb") as f:
    sample_bytes = f.read(8192)
    has_bom = sample_bytes.startswith(b'\xef\xbb\xbf')
    detected_encoding = "utf-8-sig" if has_bom else "utf-8"

# Sniff dialect using csv.Sniffer
with open(DATA_PATH, "r", encoding="utf-8", errors="replace") as f:
    sample_text = f.read(8192)
    sniffer = csv.Sniffer()
    dialect = sniffer.sniff(sample_text)
    has_header = sniffer.has_header(sample_text)

print(f"Detected Encoding: {detected_encoding}")
print(f"BOM Present: {has_bom}")
print(f"Detected Delimiter: '{dialect.delimiter}'")
print(f"Header Row Detected: {has_header}")

## 3. Dataset Loading & Shape Inspection
Load the complete CSV dataset using `pandas` without applying any data conversions.

In [ ]:
df = pd.read_csv(DATA_PATH)
n_rows, n_cols = df.shape
print(f"Dataset Dimensions: {n_rows:,} rows x {n_cols} columns")
print(f"Columns: {list(df.columns)}")

## 4. Head and Tail Inspection
Examine the first 10 and last 10 records to inspect sample records and structural alignment.

In [ ]:
print("=== FIRST 10 ROWS ===")
df.head(10)

In [ ]:
print("=== LAST 10 ROWS ===")
df.tail(10)

## 5. Column Inventory, Data Types & Missing Values
For every column, determine pandas data type, non-null count, missing value count, and missing percentage.

In [ ]:
col_summary = []
for col in df.columns:
    dtype = df[col].dtype
    non_null = int(df[col].count())
    null_cnt = int(df[col].isnull().sum())
    null_pct = (null_cnt / len(df)) * 100
    col_summary.append({
        "Column": col,
        "Pandas Dtype": str(dtype),
        "Non-Null Count": non_null,
        "Missing Count": null_cnt,
        "Missing (%)": f"{null_pct:.2f}%"
    })

pd.DataFrame(col_summary)

## 6. Duplicate Row Analysis
Check for exact duplicate rows across all columns.

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_pct = (duplicate_count / len(df)) * 100
print(f"Total exact duplicate rows: {duplicate_count} ({duplicate_pct:.2f}%)")

## 7. Numerical Column Analysis (`follower_count`)
Calculate key summary statistics: count, mean, median, standard deviation, min, max, and quartiles.

In [ ]:
fc = df["follower_count"]
stats_dict = {
    "Metric": ["Count", "Mean", "Std Dev", "Min", "25th Percentile (Q1)", "Median (50th)", "75th Percentile (Q3)", "Max", "IQR", "Skewness", "Kurtosis"],
    "Value": [
        f"{fc.count():,}",
        f"{fc.mean():.2f}",
        f"{fc.std():.2f}",
        f"{fc.min():,}",
        f"{fc.quantile(0.25):,.2f}",
        f"{fc.median():,.2f}",
        f"{fc.quantile(0.75):,.2f}",
        f"{fc.max():,}",
        f"{fc.quantile(0.75) - fc.quantile(0.25):,.2f}",
        f"{fc.skew():.4f}",
        f"{fc.kurt():.4f}"
    ]
}
pd.DataFrame(stats_dict)

In [ ]:
# Outlier and anomaly checks on follower_count
q1 = fc.quantile(0.25)
q3 = fc.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df[(fc < lower_bound) | (fc > upper_bound)]
negatives = (fc < 0).sum()
zeros = (fc == 0).sum()

print(f"Negative values count: {negatives}")
print(f"Zero values count: {zeros}")
print(f"Tukey 1.5*IQR outlier range: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Statistical outliers detected: {len(outliers)}")

## 8. Categorical & Text Column Analysis
Analyze unique counts, value frequencies, and formatting patterns across text columns (`user_id`, `location`, `language`).

In [ ]:
# Language breakdown
print("=== LANGUAGE DISTRIBUTION (10 UNIQUE CODES) ===")
lang_counts = df["language"].value_counts()
lang_df = pd.DataFrame({
    "Count": lang_counts,
    "Percentage (%)": (lang_counts / len(df) * 100).round(2)
})
lang_df

In [ ]:
# Location breakdown
print("=== LOCATION FREQUENCIES (33 UNIQUE LOCATIONS) ===")
loc_counts = df["location"].value_counts()
loc_df = pd.DataFrame({
    "Count": loc_counts,
    "Percentage (%)": (loc_counts / len(df) * 100).round(2)
})
print(loc_df.head(15))

# Format inspection: check for 'City, Country' structure
non_standard_locs = [loc for loc in loc_counts.index if "," not in loc]
print(f"\nLocations lacking ', <Country>' structure: {non_standard_locs}")

## 9. Identifier Integrity Analysis (`user_id`)
Validate uniqueness, missing values, and formatting syntax for user IDs.

In [ ]:
user_ids = df["user_id"]
total_ids = len(user_ids)
unique_ids = user_ids.nunique()
null_ids = user_ids.isnull().sum()
dup_ids = user_ids.duplicated().sum()

# Regex test for standard format: user_[8 alphanumeric characters]
valid_format_mask = user_ids.str.match(r"^user_[a-z0-9]{8}$")
invalid_format_count = (~valid_format_mask).sum()

print(f"Total IDs: {total_ids}")
print(f"Unique IDs: {unique_ids}")
print(f"Missing IDs: {null_ids}")
print(f"Duplicate IDs: {dup_ids}")
print(f"Non-conforming ID format count: {invalid_format_count}")

## 10. Date / Time Column Analysis (`account_created`)
Verify parsing capability, consistency of date formats, and temporal span.

In [ ]:
# Test parsing without modifying raw DataFrame
parsed_dates = pd.to_datetime(df["account_created"], format="%Y-%m-%d", errors="coerce")
unparseable_count = parsed_dates.isnull().sum()

print(f"Unparseable dates count (%Y-%m-%d format): {unparseable_count}")
print(f"Earliest account creation date: {parsed_dates.min().strftime('%Y-%m-%d')}")
print(f"Latest account creation date: {parsed_dates.max().strftime('%Y-%m-%d')}")
print(f"Total unique calendar days with registrations: {parsed_dates.nunique()} / 365")

# Check monthly registration volumes
monthly_counts = parsed_dates.dt.to_period("M").value_counts().sort_index()
pd.DataFrame({"Accounts Created": monthly_counts})

## 11. Data Hygiene & String Artifact Checks
Inspect all text columns for whitespace padding, control characters, empty strings, and character encoding nuances.

In [ ]:
hygiene_results = []
for col in ["user_id", "location", "language", "account_created"]:
    series = df[col].astype(str)
    leading_trailing = (series != series.str.strip()).sum()
    empty_str = (series == "").sum()
    tabs_newlines = series.str.contains(r"[\t\r\n]").sum()
    double_spaces = series.str.contains(r"  ").sum()
    
    hygiene_results.append({
        "Column": col,
        "Leading/Trailing Spaces": leading_trailing,
        "Empty Strings": empty_str,
        "Tabs/Newlines": tabs_newlines,
        "Double Spaces": double_spaces
    })

pd.DataFrame(hygiene_results)

## 12. Cross-Column Relationship Exploration
Inspect the independence between geographical location and user language preference.

In [ ]:
contingency_table = pd.crosstab(df["location"], df["language"])
print("Sample Contingency Table (Location vs Language):")
contingency_table.head(10)

## 13. Audit Summary & Deliverable Check
Confirm that the raw dataset has been thoroughly inspected without any modification.

In [ ]:
print("Audit complete. Summary results:")
print(f"- Rows: {len(df)}")
print(f"- Missing values: {df.isnull().sum().sum()}")
print(f"- Duplicate rows: {df.duplicated().sum()}")
print("- Raw dataset remains pristine and unmodified in data/raw/.")